# 02 — Prototype: Ray pipeline trên tập nhỏ

Kiểm tra pipeline Ray Data + Ray Train trên 1,000 mẫu trước khi chạy full 120k.
Mục tiêu: đảm bảo code không có lỗi, model hội tụ, benchmark cơ bản chạy được.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import ray
ray.init(ignore_reinit_error=True)
print('Ray version:', ray.__version__)
print('Cluster resources:', ray.cluster_resources())

2026-05-17 15:08:50,519	INFO worker.py:1752 -- Started a local Ray instance.


Ray version: 2.10.0
Cluster resources: {'memory': 9904339355.0, 'node:127.0.0.1': 1.0, 'object_store_memory': 4952169676.0, 'CPU': 16.0, 'node:__internal_head__': 1.0}


In [2]:
# Load nhỏ — chỉ 1000 train, 200 test
from datasets import load_dataset
import ray.data

hf = load_dataset('ag_news')
small_train = hf['train'].select(range(1000))
small_test  = hf['test'].select(range(200))

train_ds = ray.data.from_huggingface(small_train)
test_ds  = ray.data.from_huggingface(small_test)

print('Train:', train_ds.count(), '| Test:', test_ds.count())

Train: 1000 | Test: 200


In [3]:
# Tokenize
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained('bert-base-uncased')

def tokenize(batch):
    enc = tok(batch['text'], padding='max_length', truncation=True,
               max_length=128, return_tensors='np')
    return {'input_ids': enc['input_ids'],
            'attention_mask': enc['attention_mask'],
            'label': batch['label']}

import time
t0 = time.perf_counter()
train_tok = train_ds.map_batches(tokenize, batch_size=64)
n = train_tok.count()
elapsed = time.perf_counter() - t0

print(f'Tokenized {n} samples in {elapsed:.2f}s ({n/elapsed:.0f} samples/sec)')
print('Sample keys:', train_tok.take(1)[0].keys())

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "C:\Users\PC\GitHub\RAY-experiment\.venv\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [ ]:
# Prototype training — 1 epoch
import torch
from ray import train
from ray.train.torch import TorchTrainer
from ray.train import ScalingConfig

def train_loop(config):
    from transformers import AutoModelForSequenceClassification

    model = AutoModelForSequenceClassification.from_pretrained(
        'bert-base-uncased', num_labels=4
    )
    model = train.torch.prepare_model(model)
    optim = torch.optim.AdamW(model.parameters(), lr=2e-5)

    ds = train.get_dataset_shard('train')
    correct, total, loss_sum = 0, 0, 0.0

    for batch in ds.iter_torch_batches(batch_size=16, dtypes=torch.long):
        out = model(input_ids=batch['input_ids'],
                    attention_mask=batch['attention_mask'],
                    labels=batch['label'])
        optim.zero_grad()
        out.loss.backward()
        optim.step()
        correct += (out.logits.argmax(-1) == batch['label']).sum().item()
        total   += batch['label'].size(0)
        loss_sum += out.loss.item()

    train.report({'acc': correct/total, 'loss': loss_sum/total})

trainer = TorchTrainer(
    train_loop,
    train_loop_config={},
    scaling_config=ScalingConfig(num_workers=1, use_gpu=False),
    datasets={'train': train_tok},
)

result = trainer.fit()
print('Prototype result:', result.metrics)

In [ ]:
# Quick sanity check — predict 5 mẫu
LABEL_NAMES = {0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'}

samples = hf['test'].select(range(5))
for s in samples:
    inputs = tok(s['text'], return_tensors='pt', truncation=True, max_length=128)
    print(f"True: {LABEL_NAMES[s['label']]:10} | Text: {s['text'][:80]}...")

print('\nPrototype OK — sẵn sàng chạy full pipeline!')